In [1]:
import torch.nn as nn

class DNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers, fixed_output=False, linear_net=False, G=1, bias=False):
        super(DNN, self).__init__()
        self.num_layers = num_layers
        self.bias = bias

        # Define layers
        self.input_layer = nn.Linear(input_size, hidden_size, bias=bias)
        self.rnn = nn.RNN(hidden_size, hidden_size, num_layers=num_layers - 1, nonlinearity='relu', batch_first=True, bias=bias)
        self.output_layer = nn.Linear(hidden_size, output_size, bias=bias)
        if fixed_output:
            self.output_layer.requires_grad_(False)

        if linear_net:
            self.activation = nn.Identity()
        else:
            self.activation = nn.ReLU()

        # Initialize weights using Xavier with gain 0.1, and set biases to zero
        self.init_weights(fixed_output, G)

    def init_weights(self, fixed_output, G):
        nn.init.xavier_normal_(self.input_layer.weight, gain=G)
        for name, param in self.rnn.named_parameters():
            if 'weight' in name:
                nn.init.xavier_normal_(param, gain=G)
            elif 'bias' in name:
                nn.init.constant_(param, 0.0)
        if fixed_output:
            nn.init.normal_(self.output_layer.weight)
        else:
            nn.init.xavier_normal_(self.output_layer.weight, gain=G)
        if self.bias:
            nn.init.constant_(self.input_layer.bias, 0.0)
            nn.init.constant_(self.output_layer.bias, 0.0)

    def forward(self, x):
        hidden_states = []

        # Input layer
        x = self.activation(self.input_layer(x))
        hidden_states.append(x)

        # RNN layers
        x, _ = self.rnn(x.unsqueeze(1))  # Add batch dimension for RNN
        x = x.squeeze(1)  # Remove batch dimension
        hidden_states.append(x)

        # Output layer
        out = self.output_layer(x)

        return out, hidden_states

In [8]:
from copy import deepcopy
import itertools
import numpy as np
import torch
from scipy.ndimage import gaussian_filter
from torch import nn, optim

from model import DNN
from tqdm import tqdm

def one_hot(i, n):
    return np.eye(n)[i]
use_gpu = True


class Config:
    def __init__(self, **entries):
        self.gpu_id=2
        self.seed=0
        self.G=0.95
        self.hidden_size=100
        self.learning_rate=0.00001
        self.num_epochs=10000
        self.L=8
        self.input_size=100
        self.length_corridors=[10]
        self.max_move= 1
        self.min_move= 0
        self.one_hot_actions=True
        self.one_hot_inputs=True
        self.allow_backwards=True
        self.fixed_output=False
        self.egocentric_movement=False
        self.linear_net=False
        self.algo_name='ADAM'
        self.loss_fn=nn.CrossEntropyLoss()
        self.split_actions=False
        self.early_stopping=False
        self.print_progress=False
        self.corridor_dim = 1
        self.input_smoothing = 0
        self.sig_h_2 = None
        self.bias = False
        self.k = 5
        # self.__dict__.update(entries)

C = Config()

device = torch.device(f"cuda:{C.gpu_id}" if torch.cuda.is_available() and use_gpu else "cpu")
torch.manual_seed(C.seed)
np.random.seed(C.seed)
n_cors = len(C.length_corridors)
N_inputs = sum(C.length_corridors)
loss_thresh = 0.05 if not C.one_hot_inputs else 0.01
actions = np.concatenate([np.arange(-C.max_move, -C.min_move+1), np.arange(C.min_move, C.max_move + 1)])
actions = np.unique(actions)
if C.allow_backwards:
    run_actions = actions
else:
    run_actions = actions[actions >= 0]
n_actions = len(actions) + (int(C.split_actions)*(n_cors-1) * len(actions))
if C.one_hot_actions:
    actions_in = [one_hot(i, n_actions) for i in range(n_actions)]
else:
    actions_in = [np.random.normal(0, 1, size=n_actions) for i in range(n_actions)]

if C.one_hot_inputs:
    input_size = N_inputs
    output_size = N_inputs
    vecs = [np.eye(input_size)[sum(C.length_corridors[:i]):sum(C.length_corridors[:i+1])] for i in range(n_cors)]
else:
    input_size = C.input_size
    output_size = C.input_size
    vecs = [gaussian_filter(np.random.normal(size=(C.length_corridors[i]*3, C.input_size)),
                            sigma=C.length_corridors[i]*C.input_smoothing)[C.length_corridors[i]:-C.length_corridors[i]] for i in range(n_cors)]
    vecs = [vecs[i] - vecs[i].mean(axis=0) for i in range(n_cors)]
    vecs = [vec / vec.std() for vec in vecs]
    # L_vec = gaussian_filter(np.random.normal(size=(C.length_corridors[0] * 3, C.input_size)), sigma=0)[C.length_corridors[0]:2 * C.length_corridors[0]]
    # R_vec = gaussian_filter(np.random.normal(size=(C.length_corridors[1] * 3, C.input_size)), sigma=0)[C.length_corridors[1]:2 * C.length_corridors[1]]
X = []
y = []
loc_X = []
loc_y = []
corridor = []
action_taken = []
for cor, vec in enumerate(vecs):
    for loc, v in enumerate(vec):
        for a_seq in list(itertools.combinations_with_replacement(run_actions, C.k)):
            X_curr = []
            y_curr = []
            end_loc = loc
            for seq_i, a in enumerate(a_seq):
                end_loc += a
                a_i = np.where(actions == a)[0][0]
                a_i += (cor * int(C.split_actions) * (n_actions//n_cors))
                if end_loc < 0 or end_loc >= C.length_corridors[cor]:
                    break
                stim = v if seq_i == 0 else v*0
                X_curr.append(np.concatenate([stim, actions_in[a_i]]))
                y_curr.append(vec[end_loc])
            if seq_i == C.k - 1:
                X.append(X_curr)
                y.append(y_curr)
                corridor.append(cor)
                loc_X.append(loc)
                loc_y.append(end_loc)
                action_taken.append(a_seq)
X = np.array(X)
y = np.array(y)
if not C.one_hot_inputs:
    X[:, :input_size] /= X[:, :input_size].std()
    y[:, :output_size] /= y[:, :output_size].std()
corridor = np.array(corridor)
loc_X = np.array(loc_X)
loc_y = np.array(loc_y)
action_taken = np.array(action_taken)


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (163,) + inhomogeneous part.

In [7]:
X

[[],
 [array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.])],
 [array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.])],
 [array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.])],
 [array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.])],
 [array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.])],
 [array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.])],
 [array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.])],
 [array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.])],
 [array([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0.])],
 [array([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0.])],
 [array([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1.])],
 [array([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.])],
 [array([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0.])],
 [array([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1.])],
 [array([0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0.])],
 [array([0., 0., 0., 0., 0., 1., 0.

In [ ]:


X = torch.tensor(X, dtype=torch.float32).to(device)
y = torch.tensor(y, dtype=torch.float32).to(device)
# cond = (loc_X>=-min(action_taken))&(loc_X<=max(loc_X) - max(action_taken))
# X = X[cond]
# y = y[cond]
# corridor = corridor[cond]
# loc_X = loc_X[cond]
# loc_y = loc_y[cond]
# action_taken = action_taken[cond]

if C.sig_h_2:
    C.G = (C.sig_h_2*(X.shape[1]+C.hidden_size)/(2*X.shape[1]*X.var()))**(1/(2*C.L))
    print(f'Changed G to {C.G} to get sig_h_2 = {C.sig_h_2}')
# Create model
model = DNN(input_size + n_actions, C.hidden_size, output_size, C.L, C.fixed_output, C.linear_net, C.G, C.bias).to(device)
initial_weights = deepcopy(model.state_dict())
with torch.no_grad():
    outputs, hidden_states = model(X)
    print(f'Sig_2 of last hidden: {hidden_states[-1].var().item()}')

# Loss function and optimizer
criterion = C.loss_fn
algo = optim.SGD if C.algo_name == 'SGD' else optim.Adam
optimizer = algo(model.parameters(), lr=C.learning_rate)

y_var = y.var().cpu() if isinstance(criterion, nn.MSELoss) else 1
# Training loop
loss_l = []
accuracy_l = []
hidden_l = []
sample_inds = np.unique(np.linspace(0, C.num_epochs-1, 1000).astype(int))
for epoch in tqdm(range(C.num_epochs)) if C.print_progress else range(C.num_epochs):
    optimizer.zero_grad()
    outputs, hidden_states = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    loss_l.append(loss.item()/y_var)
    if C.one_hot_inputs:
        accuracy_l.append((outputs.argmax(dim=1) == y.argmax(dim=1)).float().mean().item())
        if accuracy_l[-1] == 1 and C.early_stopping:
            # print('perfect accuracy reached, stopping')
            break
    else:
        accuracy_l.append(0)
    if loss_l[-1] < loss_thresh and C.early_stopping:
        # print('loss threshold reached, stopping')
        break
    # if (epoch + 1) % int(C.num_epochs/10) == 0 and C.print_progress:
    #     print(f"Epoch {epoch + 1}/{C.num_epochs}, Loss: {loss_l[-1]:.4f}")
    if epoch in sample_inds:
        hidden_l.append([h.cpu().detach().numpy() for h in hidden_states])

# Testing
with torch.no_grad():
    outputs, hidden_states = model(X)
# print(criterion(outputs, y).item()/y_var)
